# Neural network pruning using Nonlinear Oblique Subspace ProjectionsExperiments for **CIFAR-10** supporting:> J. Pena-Campos, A. Caicedo, C. Ocampo-Martinez, D. Patino,> *"Neural Network Pruning using Nonlinear Oblique Subspace Projections"*,> ACDSA 2026, Boracay, Philippines.> DOI: [10.1109/ACDSA67686.2026.11468014](https://doi.org/10.1109/ACDSA67686.2026.11468014)The idea under test is that **functional contribution**, measured by NObSP, identifies adifferent and better set of neurons to remove than **weight magnitude** does. The notebookprunes the same network by both criteria and compares accuracy as a function of how manyneurons are removed.---### AttributionThe functions `ObSP`, `NObSP_NN_single_MultiOutput_L1` and `NObSP_NN_single_MultiOutput_L2`implement the NObSP decomposition of> A. Caicedo et al., *Nonlinear Oblique Subspace Projections*,and are **(c) Alexander Caicedo, 2023**, reproduced here with permission. See `NOTICE` atthe repository root for the licence that applies to them. Everything else in this notebookis part of the thesis work.### Before runningSet `DATA_ROOT` in the first cell to the directory holding the trained checkpoints. Thenotebooks were originally run on Google Colab; the hard-coded Drive paths have beenreplaced by that constant.The datasets are **CIFAR-10**, downloaded by `torchvision` on first run. No restricteddata is involved, and the checkpoints distributed alongside these notebooks were trainedon them.

## 1. SetupImports and device selection.

In [ ]:
# Directorio con los checkpoints. Ajustar antes de ejecutar.
DATA_ROOT = "./"

import torch
import numpy as np
import matplotlib.pyplot as plt
import os
# import scienceplots
# plt.style.use(['science', 'ieee'])

import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim

from tqdm.notebook import tqdm
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader as tf_dataloader
import pandas as pd
import seaborn as sns
from torch.utils.data import random_split

## 2. The NObSP decompositionOblique projection onto each neuron's response subspace along the reference subspace spanned by the others. Orthogonal projection is not sufficient here, because the learned subspaces overlap.

In [ ]:
def ObSP(X,Y):

    # Function to compute the oblique projection onto X along the reference space defined by Y.
    # The input data is the following:
    # X: a matriz of size Nxd, containing the basis for the subspace where the data will be projected
    # Y: a matrix of size Nxd, containing the basis for the reference subspace
    #
    # The function returns the oblique projection matrix of size NxN.

    # Converting the input data into torch tensors

    if not torch.is_tensor(X):
        X = torch.from_numpy(X)

    if not torch.is_tensor(Y):
        Y = torch.from_numpy(Y)

    N, d = X.size() # computing the size of X

    P = Y@torch.linalg.pinv(torch.t(Y)@Y)@torch.t(Y) # Conputing the orthogonal projection matriz onto the subsapce given by Y
    Q = torch.eye(N,N)-P # Computing the complement of P

    P_xy = X@torch.linalg.pinv(torch.t(X)@Q@X)@torch.t(X)@torch.t(Q) # Computing the oblique projection matriz onto X along Y

    return P_xy


def obsp_regression(X, Z, y):

    # Oblique projection using regression approach.

    # Parameters:
    # X (numpy.ndarray): Matrix representing the basis for the target subspace.
    # Z (numpy.ndarray): Matrix representing the basis for the reference subspace.
    # y (numpy.ndarray): Vector to be projected.

    # Returns:
    # beta (numpy.ndarray): Coefficients of the projection in the regression problem.


    # Compute the matrix Q
    P = Z@torch.linalg.pinv(torch.t(Z)@Z)@torch.t(Z) # Computing the orthogonal projection matriz onto the subsapce given by Y
    Q = torch.eye(Z.shape[0])-P # Computing the complement of P

    # Compute the matrix (X^T Q X)^{-1} X^T Q

    aux = torch.linalg.pinv(torch.t(X)@Q@X)@torch.t(X)@Q

    # Compute the beta coefficients
    beta = aux @ y

    return beta


def NObSP_NN_single_MultiOutput(X, y_est, model):

    # Function to decompose the output of a NN regression model using oblique subspace projections. The function computes
    # appropriate evalautions of the network that define the subspace of the nonlinear transformation of the input variables.
    # These subspaces lie in the same space where the output data is located. This function uses as input the following variables:
    #
    # X: a matrix of size Nxd, contining the input variables
    # y_est: a matrix of size Nxp, containing the estimated outputs for the input data X
    # model: the NN model using pytorch
    #
    # The function returns d oblique projection matrices of size NxN, the estimated contribution of each input variable on the output,
    # and the alpha coefificents for the out-of-sample extension

    model.eval() # Setting the model in evaluation mode
    N = np.size(X,0) # computing the size of X along dimension 0
    d = np.size(X,1) # computing the size of X along dimension 1
    p = list(model.children())[-1].out_features # computing the size of X along dimension 1
    P_xy = np.zeros((N,N,d)) # Initializing proyection matrices
    y_e = torch.from_numpy(np.zeros((N,d,p))).type(torch.float) # Initializing Matriz where the estimated nonlinear contributions will be stored, converting to a tensor object
    neurons_last = list(model.children())[-1].in_features # Obtaining the dimension of the subspace where the data lies (number of neurons in the last layer)
    Alpha = torch.zeros(neurons_last,d*p) # Initializing the matrix for the Alpha coefficients, out-of-sample extension, using the dimension where the transformed data lies

    # Computing the transformation of the input data using inference mode, and extracting the evaluation of the input on the network up to the last layer.
    with torch.inference_mode():
        y_target, X_target_tot, y_lin = model(X)

    #X_target_tot = X_target_tot-torch.mean(X_target_tot,dim=0) # Centaring the data
    P_x_target = torch.linalg.pinv(torch.t(X_target_tot)@X_target_tot)@torch.t(X_target_tot) # Finding the projection matrix onto the matrix tused to find the alpha coeficients, out-of-sample extension

    for l in range(p):
        for i in range(d):

            # Defining the input matrix that will be used to find the subspace of the nonlinear transformation of
            # the input variables x_i, onto which the output will be projected
            X_target = np.zeros((N,d))
            X_target[:,i] = X[:,i]

            # Defining the input matrix that will be used to find the reference subspace, along which the data
            # will be projected.
            X_reference = np.copy(X)
            X_reference[:,i] = 0

            # transforming the matrices to tensor objects to be used in pytorch
            X_target = torch.from_numpy(X_target).type(torch.float)
            X_reference = torch.from_numpy(X_reference).type(torch.float)

            # Computing the transformation of the input data using inference mode in order to find the basis for the susbpace of the transformations
            with torch.inference_mode():
                y_target, X_target_sub, y_lin = model(X_target) # X_target_sub is a basis for the nonlienar transformation of the data in X_target
                y_reference, X_reference_sub, y_lin = model(X_reference) # X_reference_sub is a basis for the nonlienar transformation of the data in X_reference

            # Centering the bassis of the target and reference subspaces
            X_target_sub = (X_target_sub-torch.mean(X_target_sub,dim=0))
            X_reference_sub = (X_reference_sub-torch.mean(X_reference_sub,dim=0))

            # Computing the oblique projection onto the susbspace defined by the nonlienar transformation of x_i along
            # the reference subspace, which contains the nonlinear transofrmation of all variables except x_i
            P_xy[:,:,i] = ObSP(X_target_sub,X_reference_sub)

            P = torch.from_numpy(P_xy[:,:,i]).type(torch.float) # Converting form pytorch to numpy
            y_e[:,[i],[l]] = torch.unsqueeze(P@(y_est[:,l]-torch.mean(y_est[:,l])),-1) # Using the projection matrices to ptoject the output vector and find the nonlinear contribution of each variable.

        Alpha[:,l*d:l*d+d] = torch.linalg.lstsq(X_target_tot, y_e[:,:,l].squeeze(), rcond=None, driver='gelsd')[0]
        #print(f'initial column {l*d} final column {l*d+d}')

    #print(f'Size Alpha {Alpha.size()}')
    return P_xy, y_e, Alpha

def NObSP_NN_single_MultiOutput_reg(X, y_est, model):
    model.eval()
    N, d = X.shape
    print("X: ", X.shape)
    p = list(model.children())[-1].out_features
    neurons_last = list(model.children())[-1].in_features
    betas_tensor = torch.zeros((neurons_last, d, p))
    print("beta: ", betas_tensor.shape)

    X_target = torch.zeros((N,d), dtype=torch.float)

    indices = np.argmax(F.softmax(y_est, dim=1), axis=1)
    indices = np.unique(indices)
    print(indices)

    for l in tqdm(range(p), desc="Number of tensors"):
        for i in range(d):
            if l in indices:
                X_target[:,i] = X[:,i]
                X_reference = X.clone()
                X_reference[:,i] = 0

                with torch.inference_mode():
                    _, X_target_sub, _ = model(X_target)
                    _, X_reference_sub, _ = model(X_reference)

                #print("X_target_sub: ", X_target_sub.shape)
                #print("X_reference_sub: ", X_reference_sub.shape)
                #print("y_est[:,l]: ", y_est[:,l].shape)

                X_target_sub = X_target_sub.clone()
                X_reference_sub = X_reference_sub.clone()

                X_target_sub.sub_(torch.mean(X_target_sub,dim=0))
                X_reference_sub.sub_(torch.mean(X_reference_sub,dim=0))

                beta = obsp_regression(X_target_sub, X_reference_sub, y_est[:,l]-torch.mean(y_est[:,l]))
            else:
                beta = torch.zeros(neurons_last)

            betas_tensor[:, i, l] = beta

    #betas_tensor = betas_tensor[..., indices]

    return betas_tensor

def to_categorical(y, num_classes):
    # Converting to categorical the ouput of the model
    return np.eye(num_classes)[y]

In [ ]:
def NObSP_NN_single_MultiOutput_L1(X, L_out):
  N = np.size(X,0) # computing the size of X along dimension 0
  d = np.size(X,1) # computing the size of X along dimension 1

  #print("\nDimensión 0 en X:", N)
  #print("\nDimensión 1 en X:", d)

  p = list(model.children())[-1].out_features # computing the size of X along dimension 1

  #print("\nDimensión de p:", p)

  P_xy = np.zeros((N,N,d)) # Initializing proyection matrices
  y_e = torch.from_numpy(np.zeros((N,d,p))).type(torch.float) # Initializing Matriz where the estimated nonlinear contributions will be stored, converting to a tensor object
  neurons_last = list(model.children())[-1].in_features # Obtaining the dimension of the subspace where the data lies (number of neurons in the last layer)
  Alpha = torch.zeros(neurons_last,d*p) # Initializing the matrix for the Alpha coefficients, out-of-sample extension, using the dimension where the transformed data lies

  # Computing the transformation of the input data using inference mode, and extracting the evaluation of the input on the network up to the last layer.
  #with torch.inference_mode():
  # outputs_f, L3_f, L2_f, L1_f, L_In_f = model(X)

  P_x_target = torch.linalg.pinv(torch.t(X)@X)@torch.t(X) # Finding the projection matrix onto the matrix tused to find the alpha coeficients, out-of-sample extension
  print("\nDimensión de P_x_target:", P_x_target.shape)

  for l in range(p):
    for i in range(d):
      # Defining the input matrix that will be used to find the subspace of the nonlinear transformation of
      # the input variables x_i, onto which the output will be projected
      X_target = np.zeros((N,d))
      X_target[:, i] = X[:, i].cpu().numpy()

      # Defining the input matrix that will be used to find the reference subspace, along which the data
      # will be projected.
      X_reference = X.cpu().numpy().copy()
      X_reference[:,i] = 0

      # transforming the matrices to tensor objects to be used in pytorch
      X_target = torch.from_numpy(X_target).type(torch.float)
      X_reference = torch.from_numpy(X_reference).type(torch.float)

      # Computing the transformation of the input data using inference mode in order to find the basis for the susbpace of the transformations
      with torch.inference_mode():
        y_target, X_target_sub3, X_target_sub2  = model.forward_2(X_target)
        y_reference, X_reference_sub3, X_reference_sub2 = model.forward_2(X_reference)

      # Centering the bassis of the target and reference subspaces
      X_target_sub = (X_target-torch.mean(X_target,dim=0))
      X_reference_sub = (X_reference-torch.mean(X_reference,dim=0))

      # Computing the oblique projection onto the susbspace defined by the nonlienar transformation of x_i along
      # the reference subspace, which contains the nonlinear transofrmation of all variables except x_i
      P_xy[:,:,i] = ObSP(X_target_sub,X_reference_sub)
      P = torch.from_numpy(P_xy[:,:,i]).type(torch.float) # Converting form pytorch to numpy
      y_e[:,[i],[l]] = torch.unsqueeze(P@(L_out[:,l]-torch.mean(L_out[:,l])),-1) # Using the projection matrices to ptoject the output vector and find the nonlinear contribution of each variable.

    #print(f'initial column {l*d} final column {l*d+d}')

  #print(f'Size Alpha {y_e.size()}')
  return P_xy, y_e

def NObSP_NN_single_MultiOutput_L2(X, L_out):
  N = np.size(X,0) # computing the size of X along dimension 0
  d = np.size(X,1) # computing the size of X along dimension 1

  #print("\nDimensión 0 en X:", N)
  #print("\nDimensión 1 en X:", d)

  p = list(model.children())[-1].out_features # computing the size of X along dimension 1

  #print("\nDimensión de p:", p)

  P_xy = np.zeros((N,N,d)) # Initializing proyection matrices
  y_e = torch.from_numpy(np.zeros((N,d,p))).type(torch.float) # Initializing Matriz where the estimated nonlinear contributions will be stored, converting to a tensor object
  neurons_last = list(model.children())[-1].in_features # Obtaining the dimension of the subspace where the data lies (number of neurons in the last layer)
  Alpha = torch.zeros(neurons_last,d*p) # Initializing the matrix for the Alpha coefficients, out-of-sample extension, using the dimension where the transformed data lies

  # Computing the transformation of the input data using inference mode, and extracting the evaluation of the input on the network up to the last layer.
  #with torch.inference_mode():
  # outputs_f, L3_f, L2_f, L1_f, L_In_f = model(X)

  P_x_target = torch.linalg.pinv(torch.t(X)@X)@torch.t(X) # Finding the projection matrix onto the matrix tused to find the alpha coeficients, out-of-sample extension
  print("\nDimensión de P_x_target:", P_x_target.shape)

  for l in range(p):
    for i in range(d):
      # Defining the input matrix that will be used to find the subspace of the nonlinear transformation of
      # the input variables x_i, onto which the output will be projected
      X_target = np.zeros((N,d))
      X_target[:, i] = X[:, i].cpu().numpy()

      # Defining the input matrix that will be used to find the reference subspace, along which the data
      # will be projected.
      X_reference = X.cpu().numpy().copy()
      X_reference[:,i] = 0

      # transforming the matrices to tensor objects to be used in pytorch
      X_target = torch.from_numpy(X_target).type(torch.float)
      X_reference = torch.from_numpy(X_reference).type(torch.float)

      # Computing the transformation of the input data using inference mode in order to find the basis for the susbpace of the transformations
      with torch.inference_mode():
        y_target, X_target_sub3  = model.forward_3(X_target)
        y_reference, X_reference_sub3 = model.forward_3(X_reference)

      # Centering the bassis of the target and reference subspaces
      X_target_sub = (X_target-torch.mean(X_target,dim=0))
      X_reference_sub = (X_reference-torch.mean(X_reference,dim=0))

      # Computing the oblique projection onto the susbspace defined by the nonlienar transformation of x_i along
      # the reference subspace, which contains the nonlinear transofrmation of all variables except x_i
      P_xy[:,:,i] = ObSP(X_target_sub,X_reference_sub)
      P = torch.from_numpy(P_xy[:,:,i]).type(torch.float) # Converting form pytorch to numpy
      y_e[:,[i],[l]] = torch.unsqueeze(P@(L_out[:,l]-torch.mean(L_out[:,l])),-1) # Using the projection matrices to ptoject the output vector and find the nonlinear contribution of each variable.

    #print(f'initial column {l*d} final column {l*d+d}')

  #print(f'Size Alpha {y_e.size()}')
  return P_xy, y_e

## 3. DataCIFAR-10, downloaded by `torchvision` on first run.

In [ ]:
# Cargar y preprocesar el dataset CIFAR-100

transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)

# Dividir el conjunto de entrenamiento en entrenamiento y validación
train_size = int(0.75 * len(trainset))
val_size = len(trainset) - train_size
train_subset, val_subset = random_split(trainset, [train_size, val_size])

trainloader = torch.utils.data.DataLoader(train_subset, batch_size=128,
                                          shuffle=True, num_workers=2)

valloader = torch.utils.data.DataLoader(val_subset, batch_size=128,
                                        shuffle=False, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)

testloader = torch.utils.data.DataLoader(testset, batch_size=128,
                                         shuffle=False, num_workers=2)

# Definir el número de clases
num_classes = 10

# Obtener la dimensión de la entrada aplanada
dummy_input = torch.randn(1, 3, 32, 32)
input_shape = dummy_input.view(-1).shape[0]

print("\nDimensión de la entrada aplanada:", input_shape)

In [ ]:
print(f"Tamaño total del dataset de prueba (Train): {len(trainloader.dataset)}")
print(f"Tamaño total del dataset de prueba (Validation): {len(valloader.dataset)}")
print(f"Tamaño total del dataset de prueba (Test): {len(testloader.dataset)}")

## 4. Network and trainingThe multilayer perceptron whose first linear layer, of 512 units, is the one pruned.

In [ ]:
# Definir la arquitectura de la red MLP profunda
class NetImprovedMLP(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 300)
        self.bn1 = nn.BatchNorm1d(300)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(300, 150)
        self.bn2 = nn.BatchNorm1d(150)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)
        self.fc3 = nn.Linear(150, 50)
        self.bn3 = nn.BatchNorm1d(50)
        self.relu3 = nn.ReLU()
        self.fc4 = nn.Linear(50, num_classes)

    def forward(self, x):
        x = x.view(x.size(0),-1)
        x1 = self.dropout1(self.relu1(self.bn1(self.fc1(x))))
        x2 = self.dropout2(self.relu2(self.bn2(self.fc2(x1))))
        x3 = self.relu3(self.bn3(self.fc3(x2)))
        x4 = self.fc4(x3)
        prob = F.softmax(x4, dim=1)
        return prob, x4, x3, x2, x1, x

    def forward_2(self, x1):
        x2 = self.dropout2(self.relu2(self.bn2(self.fc2(x1))))
        x3 = self.relu3(self.bn3(self.fc3(x2)))
        x4 = self.fc4(x3)
        return x4, x3, x2

    def forward_3(self, x2):
        x3 = self.relu3(self.bn3(self.fc3(x2)))
        x4 = self.fc4(x3)
        return x4, x3

model = NetImprovedMLP(input_shape, num_classes)

In [ ]:
# Definir la función de pérdida y el optimizador
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Entrenar el modelo
num_epochs = 100  # Un número máximo de épocas
patience = 10     # Número de épocas sin mejora para detener
best_val_loss = float('inf')
epochs_no_improve = 0
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

for epoch in range(num_epochs):
    # Entrenamiento
    model.train()
    running_loss = 0.0
    correct_predictions_train = 0
    total_samples_train = 0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data
        optimizer.zero_grad()
        outputs, _, _, _, _, _ = model.forward(inputs) # Desempaquetar las salidas
        # outputs, layer3_out, layer2_out, layer1_out, input_flattened = model(inputs) # Desempaquetar las salidas
        #outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total_samples_train += labels.size(0)
        correct_predictions_train += (predicted == labels).sum().item()

    epoch_loss_train = running_loss / len(trainloader)
    epoch_accuracy_train = correct_predictions_train / total_samples_train
    train_losses.append(epoch_loss_train)
    train_accuracies.append(epoch_accuracy_train)
    print(f'Época {epoch + 1}, Pérdida entrenamiento: {epoch_loss_train:.4f}, Precisión entrenamiento: {epoch_accuracy_train:.4f}', end=" ")

    # Validación
    model.eval()
    running_val_loss = 0.0
    correct_predictions_val = 0
    total_samples_val = 0
    with torch.no_grad():
        for data in valloader:
            inputs, labels = data
            outputs, _, _, _, _, _ = model.forward(inputs) # Desempaquetar las salidas (solo necesitamos outputs para la validación)
            # outputs = model(inputs)
            val_loss = criterion(outputs, labels)
            running_val_loss += val_loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total_samples_val += labels.size(0)
            correct_predictions_val += (predicted == labels).sum().item()

    epoch_loss_val = running_val_loss / len(valloader)
    epoch_accuracy_val = correct_predictions_val / total_samples_val
    val_losses.append(epoch_loss_val)
    val_accuracies.append(epoch_accuracy_val)
    print(f'Pérdida validación: {epoch_loss_val:.4f}, Precisión validación: {epoch_accuracy_val:.4f}')

    # Early stopping
    if epoch_loss_val < best_val_loss:
        best_val_loss = epoch_loss_val
        epochs_no_improve = 0
        best_model_state = model.state_dict() # Guardar el mejor estado del modelo
    else:
        epochs_no_improve += 1
        if epochs_no_improve == patience:
            print(f'Early stopping activado en la época {epoch + 1}')
            model.load_state_dict(best_model_state) # Cargar el mejor estado del modelo
            break

In [ ]:
# Evaluar el modelo
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data
        # outputs = model(images)
        L_out, L4, L3, L2, L1, L_In = model(images) # Desempaquetar las salidas
        _, predicted = torch.max(L_out.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'\nPrecisión del modelo en el conjunto de prueba: {100 * correct / total:.2f} %')

ruta_guardado_completo = DATA_ROOT + 'modelo_completo_Ori.pth'

# Guardar el modelo completo
torch.save(model, ruta_guardado_completo)

print(f"Modelo completo guardado exitosamente en: {ruta_guardado_completo}")

In [ ]:
# Graficar la pérdida y la precisión durante el entrenamiento
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(range(1, len(train_losses) + 1), train_losses, label='Pérdida de entrenamiento')
plt.plot(range(1, len(val_losses) + 1), val_losses, label='Pérdida de validación')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.title('Pérdida durante el entrenamiento y la validación')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, len(train_accuracies) + 1), train_accuracies, label='Precisión de entrenamiento')
plt.plot(range(1, len(val_accuracies) + 1), val_accuracies, label='Precisión de validación')
plt.xlabel('Época')
plt.ylabel('Precisión')
plt.title('Precisión durante el entrenamiento y la validación')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Cargar el modelo completo
try:
    model = torch.load(DATA_ROOT + "modelo_completo_Ori.pth", weights_only=False)
    print("Modelo cargado exitosamente.")

    # Si vas a usar el modelo para inferencia, es recomendable ponerlo en modo de evaluación
    model.eval()
    print("Modelo puesto en modo de evaluación.")

except FileNotFoundError:
    print(f"Error: No se encontró el archivo en la ruta: {ruta_guardado_completo}")
except Exception as e:
    print(f"Ocurrió un error al cargar el modelo: {e}")

## 5. Extracting the layer responsesThe decomposition needs the responses of the target layer over a fixed subset of inputs, and the weights of the layer that follows.

In [ ]:
# Acceder a los pesos de la capa

first_linear_layer = model.fc1
weights_layer1 = first_linear_layer.weight.data.numpy()
biases_layer1 = first_linear_layer.bias.data.numpy()

print("\nForma de los pesos de la primera capa lineal (300 neuronas):", weights_layer1.shape)
print("Forma de los biases de la primera capa lineal (300 neuronas):", biases_layer1.shape)

first_linear_layer = model.fc2
weights_layer1 = first_linear_layer.weight.data.numpy()
biases_layer1 = first_linear_layer.bias.data.numpy()

print("\nForma de los pesos de la primera capa lineal (150 neuronas):", weights_layer1.shape)
print("Forma de los biases de la primera capa lineal (150 neuronas):", biases_layer1.shape)

first_linear_layer = model.fc3
weights_layer1 = first_linear_layer.weight.data.numpy()
biases_layer1 = first_linear_layer.bias.data.numpy()

print("\nForma de los pesos de la primera capa lineal (50 neuronas):", weights_layer1.shape)
print("Forma de los biases de la primera capa lineal (50 neuronas):", biases_layer1.shape)

# Imprimir algunos valores de los pesos (opcional - comentar para evitar salida larga)
# print("\nAlgunos valores de los pesos de la primera capa lineal:\n", weights_layer1[:5, :5])


In [ ]:
from torch.utils.data import Subset

num_images = 600
indices = list(range(num_images))
subset = Subset(testset, indices)

# Inicializar listas para almacenar las imágenes y las etiquetas
images_list = []
labels_list = []

# Iterar a través del subset y agregar las imágenes y etiquetas a las listas
for img, label in subset:
    images_list.append(img)
    labels_list.append(label)

# Convertir las listas de imágenes y etiquetas a tensores de PyTorch
all_images = torch.stack(images_list)
all_labels = torch.tensor(labels_list)

print("Forma del tensor de todas las imágenes:", all_images.shape)
print("Forma del tensor de todas las etiquetas:", all_labels.shape)

In [ ]:
model = torch.load(DATA_ROOT + "modelo_completo_Ori.pth", weights_only=False)
# Evaluar el modelo
model.eval()
correct = 0
total = 0
with torch.no_grad():
  L_out, L4, L3, L2, L1, L_In = model(all_images) # Desempaquetar las salidas
  _, predicted = torch.max(L_out.data, 1)
  total += all_labels.size(0)
  correct += (predicted == all_labels).sum().item()

print(f'\nPrecisión del modelo en el conjunto de prueba: {100 * correct / total:.2f} %')

In [ ]:
print("\nForma de los pesos de la capa 1 lineal (512 neuronas):", L_In.shape)
print("\nForma de los pesos de la capa 2 lineal (512 neuronas):", L1.shape)
print("\nForma de los pesos de la capa 3 lineal (512 neuronas):", L2.shape)
print("\nForma de los pesos de la capa 4 lineal (512 neuronas):", L3.shape)
print("\nForma de los pesos de la capa 4 lineal (512 neuronas):", L4.shape)
print("\nForma de los pesos de la capa 4 lineal (512 neuronas):", L_out.shape)

## 6. Contribution of each neuronEach neuron's own component of the output, from which the ranking follows.

In [ ]:
Pxy_L1, y_e_L1 = NObSP_NN_single_MultiOutput_L1(L1, L_out)
Pxy_L2, y_e_L2 = NObSP_NN_single_MultiOutput_L2(L2, L_out)

In [ ]:
root_N = DATA_ROOT + ''
nombre_archivo = "Proyecciones_YEst_CIFAR10.pkl"

with open(root_N+nombre_archivo, 'wb') as archivo:
  pickle.dump([Pxy_L1, y_e_L1, Pxy_L2, y_e_L2], archivo)

## 7. Per-image inspectionWhat the decomposition says about individual examples, before aggregating across the dataset.

In [ ]:
Im_N=156
plt.imshow(Pxy_L1[:, :, Im_N], cmap='gray')
plt.axis('off')  # Para no mostrar los ejes
plt.title('Imagen Pxy_L1[:, :, '+ str(Im_N)+'] (Escala de Grises)')
plt.show()

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

x = np.linspace(0, Pxy_L1[:, :, Im_N].shape[1] - 1, Pxy_L1[:, :, Im_N].shape[1])
y = np.linspace(0, Pxy_L1[:, :, Im_N].shape[0] - 1, Pxy_L1[:, :, Im_N].shape[0])
X, Y = np.meshgrid(x, y)

surf = ax.plot_surface(X, Y, Pxy_L1[:, :, Im_N], cmap='viridis')

# 4. Añadir etiquetas y título
ax.set_xlabel('Columna')
ax.set_ylabel('Fila')
ax.set_zlabel('Valor')
ax.set_title('Gráfica de Superficie de la Matriz')

# 5. Añadir una barra de color (opcional)
#fig.colorbar(surf, shrink=0.5, aspect=10)

# Establecer los ángulos de visión
elev = 30   # Ángulo de elevación en grados (arriba/abajo)
azim = 45   # Ángulo de azimut en grados (izquierda/derecha)
ax.view_init(elev=elev, azim=azim)

# 6. Mostrar la gráfica
plt.show()

import imageio.v3 as iio

video_filename = 'video_Pxy_L1.mp4'

# Lista para almacenar los frames
frames = []

# Iterar a través de las 300 "imágenes" (capas)
for i in range(Pxy_L1.shape[2]):
    # Extraer la imagen actual
    imagen = Pxy_L1[:, :, i]

    # Crear una figura y un eje para mostrar la imagen
    fig, ax = plt.subplots()
    ax.imshow(imagen, cmap='gray')  # Puedes cambiar 'gray' a otro colormap si lo deseas
    ax.axis('off')  # Ocultar los ejes

    # Dibujar la figura en un canvas y obtener los datos de la imagen
    fig.canvas.draw()
    frame = np.array(fig.canvas.renderer.buffer_rgba())

    # Añadir el frame a la lista de frames (convertir de RGBA a RGB)
    frames.append(frame[:, :, :3])

    # Cerrar la figura para liberar memoria
    plt.close(fig)

# Guardar los frames como un video usando imageio
iio.imwrite(video_filename, np.array(frames), fps=10)  # fps define la velocidad del video (frames por segundo)

print(f'Video guardado como {video_filename}')

In [ ]:
Im_N=6
plt.imshow(y_e_L1[:, :, Im_N], cmap='gray')
plt.axis('off')  # Para no mostrar los ejes
plt.title('y_e_L1[:, :, '+ str(Im_N)+'] (Escala de Grises)')
plt.show()

In [ ]:
Cont_NL1_sorted = torch.from_numpy(np.zeros((300,2,10))).type(torch.float)
Cont_NL2_sorted = torch.from_numpy(np.zeros((150,2,10))).type(torch.float)

indices_L1 = torch.arange(1, Cont_NL1_sorted.shape[0]+1)  # Crear un tensor de índices del 1 al 300
indices_L2 = torch.arange(1, Cont_NL2_sorted.shape[0]+1)  # Crear un tensor de índices del 1 al 300

for i in range(Cont_NL1_sorted.shape[2]):
  Cont_N=torch.diag(torch.t(y_e_L1[:,:,i])@y_e_L1[:,:,i])/torch.max(torch.diag(torch.t(y_e_L1[:,:,i])@y_e_L1[:,:,i]))
  Cont_Ni = torch.stack((indices_L1.float(), Cont_N), dim=1) # Ahora Cont_N_with_indices tiene forma [300, 2], donde la primera columna son los índices y la segunda son los valores de Cont_N

  # Organizar de mayor a menor según Cont_N sin perder el índice original
  # Obtener los índices ordenados por el valor de Cont_N (segunda columna)
  sorted_indices = torch.argsort(Cont_N, descending=True)

  # Reorganizar Cont_N_with_indices usando los índices ordenados
  Cont_NL1_sorted[:,:,i] = Cont_Ni[sorted_indices]

  # Graficar la variable ordenada y los índices
  # Extraer los índices y los valores ordenados para graficar
  sorted_indices_for_plot = Cont_NL1_sorted[:, 0, i]
  sorted_values_for_plot = Cont_NL1_sorted[:, 1, i]

  # Crear la gráfica
  plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
  plt.plot(indices_L1.numpy(), sorted_values_for_plot.numpy(), marker='o', linestyle='-')
  plt.xlabel('Índice Original')
  plt.ylabel('Valor de Contribución Neurona')
  plt.title('Valores de Contribución N=300 Clase: '+str(i))
  plt.xticks(sorted_indices_for_plot.numpy())  # Establecer los ticks del eje X como los índices
  plt.grid(True)
  plt.show()

  Cont_N=torch.diag(torch.t(y_e_L2[:,:,i])@y_e_L2[:,:,i])/torch.max(torch.diag(torch.t(y_e_L2[:,:,i])@y_e_L2[:,:,i]))
  Cont_Ni = torch.stack((indices_L2.float(), Cont_N), dim=1) # Ahora Cont_N_with_indices tiene forma [300, 2], donde la primera columna son los índices y la segunda son los valores de Cont_N

  # Organizar de mayor a menor según Cont_N sin perder el índice original
  # Obtener los índices ordenados por el valor de Cont_N (segunda columna)
  sorted_indices = torch.argsort(Cont_N, descending=True)

  # Reorganizar Cont_N_with_indices usando los índices ordenados
  Cont_NL2_sorted[:,:,i] = Cont_Ni[sorted_indices]

  # Graficar la variable ordenada y los índices
  # Extraer los índices y los valores ordenados para graficar
  sorted_indices_for_plot = Cont_NL2_sorted[:, 0, i]
  sorted_values_for_plot = Cont_NL2_sorted[:, 1, i]

  # Crear la gráfica
  plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
  plt.plot(indices_L2.numpy(), sorted_values_for_plot.numpy(), marker='o', linestyle='-')
  plt.xlabel('Índice Original')
  plt.ylabel('Valor de Contribución Neurona')
  plt.title('Valores de Contribución N=150 Clase: '+str(i))
  plt.xticks(sorted_indices_for_plot.numpy())  # Establecer los ticks del eje X como los índices
  plt.grid(True)
  plt.show()

In [ ]:
# Crear la matriz de concurrencia
concurrencia_matrix_L1 = np.zeros((300), dtype=float)
neu_i=0
# Llenar la matriz de concurrencia
for neu_i in range(300):
    for pos_i in range(10):
        # Contar cuántas veces la neurona neu_i aparece en la posición pos_i para todas las clases
        concurrencia_matrix_L1[neu_i]=concurrencia_matrix_L1[neu_i].item()+Cont_NL1_sorted[Cont_NL1_sorted[:,0,pos_i] == neu_i+1,1,pos_i].item()

print("Forma de la matriz de concurrencia:", concurrencia_matrix_L1.shape)
#print(concurrencia_matrix_L1)

plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
plt.stem(indices_L1.numpy(), concurrencia_matrix_L1, linefmt='-', markerfmt='o', basefmt=' ')
plt.xlabel('Índice Original')
plt.ylabel('Valor de Contribución Neurona')
plt.title('Valores de Contribución N=300')
plt.grid(True)
plt.show()

# Crear la matriz de concurrencia
concurrencia_matrix_L2 = np.zeros((150), dtype=float)

# Llenar la matriz de concurrencia
for neu_i in range(150):
    for pos_i in range(10):
        # Contar cuántas veces la neurona neu_i aparece en la posición pos_i para todas las clases
        concurrencia_matrix_L2[neu_i]=concurrencia_matrix_L2[neu_i].item()+Cont_NL1_sorted[Cont_NL1_sorted[:,0,pos_i] == neu_i+1,1,pos_i].item()

print("Forma de la matriz de concurrencia:", concurrencia_matrix_L2.shape)
#print(concurrencia_matrix_L1)

plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
plt.stem(indices_L2.numpy(), concurrencia_matrix_L2, linefmt='-', markerfmt='o', basefmt=' ')
plt.xlabel('Índice Original')
plt.ylabel('Valor de Contribución Neurona')
plt.title('Valores de Contribución N=150')
plt.grid(True)
plt.show()

In [ ]:
print("El valor minimo es: ",np.min(concurrencia_matrix_L1))
print("Esta ubicado en la posición: ",np.argmin(concurrencia_matrix_L1))
print("Verificando el resultado: ",concurrencia_matrix_L1[np.argmin(concurrencia_matrix_L1)])

print("El valor minimo es: ",np.min(concurrencia_matrix_L2))
print("Esta ubicado en la posición: ",np.argmin(concurrencia_matrix_L2))
print("Verificando el resultado: ",concurrencia_matrix_L2[np.argmin(concurrencia_matrix_L2)])

# Reorganizar Cont_N_with_indices usando los índices ordenados
indices_ordenamiento = np.argsort(concurrencia_matrix_L2)[::-1]
indices_ordenamiento[-1]
np.min(indices_ordenamiento)
np.max(indices_ordenamiento)
contribuciones_ordenadas = concurrencia_matrix_L2[indices_ordenamiento]

plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
plt.plot(contribuciones_ordenadas, marker='o', linestyle='-')
plt.xlabel('Neurona')
plt.ylabel('Contribución')
plt.title('Valores de Contribución N=150')
plt.xticks(indices_ordenamiento)  # Establecer los ticks del eje X como los índices
plt.grid(True)
plt.show()

suma_acumulada = np.cumsum(contribuciones_ordenadas)
print("Suma acumulada:", suma_acumulada[-1])
print("Suma total:", np.sum(contribuciones_ordenadas))

plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
plt.plot(suma_acumulada, marker='o', linestyle='-')
plt.xlabel('Neurona')
plt.ylabel('Contribución')
plt.title('Suma acumulada N=150')
plt.xticks(indices_ordenamiento)  # Establecer los ticks del eje X como los índices
plt.grid(True)
plt.show()

Por_Dej=0.8

print("Suma total:", Por_Dej*np.sum(contribuciones_ordenadas))
suma_acumulada>=Por_Dej*np.sum(contribuciones_ordenadas)
indices_ordenamiento[np.where(suma_acumulada>=Por_Dej*np.sum(contribuciones_ordenadas))[0]]

print("La longitud de posiciones inferiores al 90% es: ",len(np.where(suma_acumulada>=Por_Dej*np.sum(contribuciones_ordenadas))[0]))

indices_ordenamiento_L1 = np.argsort(concurrencia_matrix_L1)[::-1]
indices_ordenamiento_L1[-1]
np.min(indices_ordenamiento_L1)
np.max(indices_ordenamiento_L1)
contribuciones_ordenadas_L1 = concurrencia_matrix_L1[indices_ordenamiento_L1]

plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
plt.plot(contribuciones_ordenadas_L1, marker='o', linestyle='-')
plt.xlabel('Neurona')
plt.ylabel('Contribución')
plt.title('Valores de Contribución N=300')
plt.xticks(indices_ordenamiento_L1)  # Establecer los ticks del eje X como los índices
plt.grid(True)
plt.show()

suma_acumulada_L1 = np.cumsum(contribuciones_ordenadas_L1)
print("Suma acumulada:", suma_acumulada_L1[-1])
print("Suma total:", np.sum(contribuciones_ordenadas_L1))

plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
plt.plot(suma_acumulada_L1, marker='o', linestyle='-')
plt.xlabel('Neurona')
plt.ylabel('Contribución')
plt.title('Suma acumulada N=300')
plt.xticks(indices_ordenamiento_L1)  # Establecer los ticks del eje X como los índices
plt.grid(True)
plt.show()

print("Suma total:", Por_Dej*np.sum(contribuciones_ordenadas_L1))
suma_acumulada_L1>=Por_Dej*np.sum(contribuciones_ordenadas_L1)
indices_ordenamiento_L1[np.where(suma_acumulada_L1>=Por_Dej*np.sum(contribuciones_ordenadas_L1))[0]]

print("La longitud de posiciones inferiores al 90% es: ",len(np.where(suma_acumulada_L1>=Por_Dej*np.sum(contribuciones_ordenadas_L1))[0]))

## 8. Removing a neuron`Quitar_Contri` zeroes a neuron so that accuracy can be tracked as neurons are removed.

In [ ]:
def Quitar_Contri(model_Modif, indice, model_Ori, capa):
  print("Se quita el indice: ",indice)

  if capa == 2:
    first_linear_layer = model_Modif.fc2
    N = '150'
  elif capa ==1:
    first_linear_layer = model_Modif.fc1
    N = '300'

  weights_layer1 = first_linear_layer.weight.data.numpy()
  biases_layer1 = first_linear_layer.bias.data.numpy()

  print("\nForma de los pesos de la primera capa lineal ("+N+" neuronas):", weights_layer1.shape)
  plt.figure(figsize=(30, 6))
  plt.imshow(weights_layer1, cmap='gray')
  plt.axis('off')  # Para no mostrar los ejes
  plt.title('Pesos')
  plt.show()

  print("Forma de los biases de la primera capa lineal ("+N+" neuronas):", biases_layer1.shape)
  plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
  plt.plot(biases_layer1, marker='o', linestyle='-')
  plt.grid(True)
  plt.show()

  weights_layer1[indice,:] = 0
  biases_layer1[indice] = 0

  if capa == 2:
    model_Modif.fc2.weight.data = torch.from_numpy(weights_layer1)
    model_Modif.fc2.bias.data = torch.from_numpy(biases_layer1)
  elif capa ==1:
    model_Modif.fc1.weight.data = torch.from_numpy(weights_layer1)
    model_Modif.fc1.bias.data = torch.from_numpy(biases_layer1)


  print("\nForma de los pesos de la primera capa lineal ("+N+" neuronas):", weights_layer1.shape)
  plt.figure(figsize=(30, 6))
  plt.imshow(weights_layer1, cmap='gray')
  plt.axis('off')  # Para no mostrar los ejes
  plt.title('Pesos')
  plt.show()
  print("Forma de los biases de la primera capa lineal ("+N+" neuronas):", biases_layer1.shape)
  plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
  plt.plot(biases_layer1, marker='o', linestyle='-')
  plt.grid(True)
  plt.show()

  # Evaluar el modelo
  model_Ori.eval()
  correct_r = 0
  total = 0
  with torch.no_grad():
    L_out, L4, L3, L2, L1, L_In = model_Ori(all_images) # Desempaquetar las salidas
    _, predicted = torch.max(L_out.data, 1)
    total += all_labels.size(0)
    correct_r += (predicted == all_labels).sum().item()

  print(f'\nPrecisión del modelo en el conjunto de prueba: {100 * correct_r / total:.2f} %')

  # Evaluar el modelo
  model_Modif.eval()
  correct = 0
  total = 0
  with torch.no_grad():
    L_out, L4, L3, L2, L1, L_In = model_Modif(all_images) # Desempaquetar las salidas
    _, predicted = torch.max(L_out.data, 1)
    total += all_labels.size(0)
    correct += (predicted == all_labels).sum().item()

  print(f'\nPrecisión del modelo en el conjunto de prueba: {100 * correct / total:.2f} %')

  return model_Modif, 100 * correct / total, 100 * correct_r / total

In [ ]:
import copy

In [ ]:
model = torch.load(DATA_ROOT + "modelo_completo_Ori.pth", weights_only=False)
model_mod = copy.deepcopy(model)

Perf_L2 = np.zeros((len(np.where(suma_acumulada>=0.9*np.sum(contribuciones_ordenadas))[0])), dtype=float)
cont=0

for i_ind in indices_ordenamiento[np.where(suma_acumulada>=0.9*np.sum(contribuciones_ordenadas))[0]]:
  model_mod, Perf_L2[cont], Perf_L2_Org = Quitar_Contri(model_mod, i_ind, model, 2)
  cont+=1
Ind=np.arange(150, 150-(len(np.where(suma_acumulada>=0.9*np.sum(contribuciones_ordenadas))[0])+1), -1)
plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
plt.stem(np.arange(0, (len(np.where(suma_acumulada>=0.9*np.sum(contribuciones_ordenadas))[0])+1), 1),np.insert(Perf_L2, 0, Perf_L2_Org), linefmt='-', markerfmt='o', basefmt=' ')
plt.xlabel('Neuronas')
plt.ylabel('Desempeño')
plt.title('Pruning de N=150')
plt.xticks(np.arange(0, len(Ind), 1), Ind)
plt.grid(True)
plt.show()

model_mod1 = copy.deepcopy(model)
model_mod1, Perf_L2_1, Perf_L2_Org  = Quitar_Contri(model_mod1, indices_ordenamiento[np.where(suma_acumulada>=0.9*np.sum(contribuciones_ordenadas))[0]], model, 2)

#print("Se quita el indice: ",indices_ordenamiento[-1])

#first_linear_layer = model_mod.fc2
#weights_layer1 = first_linear_layer.weight.data.numpy()
#biases_layer1 = first_linear_layer.bias.data.numpy()

#print("\nForma de los pesos de la primera capa lineal (150 neuronas):", weights_layer1.shape)
#print(weights_layer1[indices_ordenamiento[-1]-4:indices_ordenamiento[-1]+1,:])
#print("Forma de los biases de la primera capa lineal (150 neuronas):", biases_layer1.shape)
#plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
#plt.plot(biases_layer1, marker='o', linestyle='-')
#plt.grid(True)
#plt.show()

#first_linear_layer = model_mod.fc2
#weights_layer1 = first_linear_layer.weight.data.numpy()
#weights_layer1[indices_ordenamiento[-1],:] = 0
#biases_layer1 = first_linear_layer.bias.data.numpy()
#biases_layer1[indices_ordenamiento[-1]] = 0

#model_mod.fc2.weight.data = torch.from_numpy(weights_layer1)
#model_mod.fc2.bias.data = torch.from_numpy(biases_layer1)

#print("\nForma de los pesos de la primera capa lineal (150 neuronas):", weights_layer1.shape)
#print(weights_layer1[indices_ordenamiento[-1]-4:indices_ordenamiento[-1]+1,:])
#print("Forma de los biases de la primera capa lineal (150 neuronas):", biases_layer1.shape)
#plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
#plt.plot(biases_layer1, marker='o', linestyle='-')
#plt.grid(True)
#plt.show()

In [ ]:
model = torch.load(DATA_ROOT + "modelo_completo_Ori.pth", weights_only=False)
model_mod2 = copy.deepcopy(model)

Perf_L1 = np.zeros((len(np.where(suma_acumulada_L1>=0.9*np.sum(contribuciones_ordenadas_L1))[0])), dtype=float)
cont=0

for i_ind in indices_ordenamiento_L1[np.where(suma_acumulada_L1>=0.9*np.sum(contribuciones_ordenadas_L1))[0]]:
  model_mod2, Perf_L1[cont], Perf_L1_Org = Quitar_Contri(model_mod2, i_ind, model, 1)
  cont+=1

Ind=np.arange(300, 300-(len(np.where(suma_acumulada_L1>=0.9*np.sum(contribuciones_ordenadas_L1))[0])+1), -1)
plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
plt.stem(np.arange(0, (len(np.where(suma_acumulada_L1>=0.9*np.sum(contribuciones_ordenadas_L1))[0])+1), 1),np.insert(Perf_L1, 0, Perf_L1_Org), linefmt='-', markerfmt='o', basefmt=' ')
plt.xlabel('Neuronas')
plt.ylabel('Desempeño')
plt.title('Pruning de N=300')
plt.xticks(np.arange(0, len(Ind), 1), Ind)
plt.grid(True)
plt.show()

model_mod3 = copy.deepcopy(model)
model_mod3, Perf_L1_1, Perf_L1_Org  = Quitar_Contri(model_mod3, indices_ordenamiento_L1[np.where(suma_acumulada_L1>=0.9*np.sum(contribuciones_ordenadas_L1))[0]], model, 1)

In [ ]:
def Quitar_Contri_NVerbo(model_Mod, indice, capa):
  if capa == 2:
    first_linear_layer = model_Mod.fc2
    N = '150'
  elif capa ==1:
    first_linear_layer = model_Mod.fc1
    N = '300'

  weights_layer1 = first_linear_layer.weight.data.numpy()
  biases_layer1 = first_linear_layer.bias.data.numpy()

  weights_layer1[indice,:] = 0
  biases_layer1[indice] = 0

  if capa == 2:
    model_Mod.fc2.weight.data = torch.from_numpy(weights_layer1)
    model_Mod.fc2.bias.data = torch.from_numpy(biases_layer1)
  elif capa ==1:
    model_Mod.fc1.weight.data = torch.from_numpy(weights_layer1)
    model_Mod.fc1.bias.data = torch.from_numpy(biases_layer1)

  # Evaluar el modelo
  model_Mod.eval()
  correct = 0
  total = 0
  with torch.no_grad():
    L_out, L4, L3, L2, L1, L_In = model_Mod(all_images) # Desempaquetar las salidas
    _, predicted = torch.max(L_out.data, 1)
    total += all_labels.size(0)
    correct += (predicted == all_labels).sum().item()

  return model_Mod, 100 * correct / total

In [ ]:
model = torch.load(DATA_ROOT + "modelo_completo_Ori.pth", weights_only=False)
model_mod4 = copy.deepcopy(model)

model_mod4, Perf_L1_2, Perf_L1_Org1  = Quitar_Contri(model_mod4, indices_ordenamiento_L1[np.where(suma_acumulada_L1>=Por_Dej*np.sum(contribuciones_ordenadas_L1))[0]], model, 1)
model_mod4, Perf_L2_2, Perf_L2_Org1  = Quitar_Contri(model_mod4, indices_ordenamiento[np.where(suma_acumulada>=Por_Dej*np.sum(contribuciones_ordenadas))[0]], model, 2)
len(indices_ordenamiento[np.where(suma_acumulada>=Por_Dej*np.sum(contribuciones_ordenadas))[0]])

In [ ]:
model = torch.load(DATA_ROOT + "modelo_completo_Ori.pth", weights_only=False)
model_mod5 = copy.deepcopy(model)

cont_i = 0
cont_j = 0
Perf_L1_L2 = np.zeros((len(range(0, -101, -1)), len(range(0, -151, -1))), dtype=float)
indices_ordenamiento_L1[-1]
for i_N in range(0, -101, -1):
  print("Voy en: ", i_N)
  if i_N !=0:
    model = torch.load(DATA_ROOT + "modelo_completo_Ori.pth", weights_only=False)
    first_linear_layer = model.fc1
    weights_layer1 = first_linear_layer.weight.data.numpy()
    biases_layer1 = first_linear_layer.bias.data.numpy()
    model_mod5.fc1.weight.data = torch.from_numpy(weights_layer1)
    model_mod5.fc1.bias.data = torch.from_numpy(biases_layer1)
    model_mod5, Perf_L1_L2[cont_i,cont_j] = Quitar_Contri_NVerbo(model_mod5, indices_ordenamiento[i_N], 2)
  for j_N in range(0, -151, -1):
    if i_N==-53 and j_N==0:
      plt.figure(figsize=(30, 6))
      plt.imshow(model_mod5.fc1.weight.data.numpy(), cmap='gray')
      plt.axis('off')  # Para no mostrar los ejes
      plt.title('Pesos')
      plt.show()

      plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
      plt.plot(model_mod5.fc1.bias.data.numpy(), marker='o', linestyle='-')
      plt.grid(True)
      plt.show()

      plt.figure(figsize=(30, 6))
      plt.imshow(model_mod5.fc2.weight.data.numpy(), cmap='gray')
      plt.axis('off')  # Para no mostrar los ejes
      plt.title('Pesos')
      plt.show()

      plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
      plt.plot(model_mod5.fc2.bias.data.numpy(), marker='o', linestyle='-')
      plt.grid(True)
      plt.show()

    if i_N == 0 and j_N == 0:
      model.eval()
      correct = 0
      total = 0
      with torch.no_grad():
        L_out, L4, L3, L2, L1, L_In = model(all_images) # Desempaquetar las salidas
        _, predicted = torch.max(L_out.data, 1)
        total += all_labels.size(0)
        correct += (predicted == all_labels).sum().item()
      Perf_L1_L2[cont_i,cont_j]= 100 * correct / total
    else:
      model_mod5, Perf_L1_L2[cont_i,cont_j] = Quitar_Contri_NVerbo(model_mod5, indices_ordenamiento_L1[j_N], 1)
    cont_j+=1
  cont_i+=1
  cont_j=0

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

x = np.linspace(0, Perf_L1_L2.shape[1] - 1, Perf_L1_L2.shape[1])
y = np.linspace(0, Perf_L1_L2.shape[0] - 1, Perf_L1_L2.shape[0])
X, Y = np.meshgrid(x, y)

surf = ax.plot_surface(X, Y, Perf_L1_L2, cmap='viridis')

# 4. Añadir etiquetas y título
ax.set_xlabel('N 300')
ax.set_ylabel('N 150')
ax.set_zlabel('Performance')

# 5. Añadir una barra de color (opcional)
#fig.colorbar(surf, shrink=0.5, aspect=10)

# Establecer los ángulos de visión
elev = 15   # Ángulo de elevación en grados (arriba/abajo)
azim = 20   # Ángulo de azimut en grados (izquierda/derecha)
ax.view_init(elev=elev, azim=azim)

# 6. Mostrar la gráfica
plt.show()

Perf_L1_L2

nombre_archivo = DATA_ROOT + 'perf_prun_NObSP.png'
plt.savefig(nombre_archivo, dpi=600, bbox_inches='tight')

## 9. The comparisonNObSP ranking against L1 and L2 magnitude pruning, accuracy versus neurons removed.

In [ ]:
import torch.nn.utils.prune as prune

cont_i = 0
cont_j = 0
Perf_L1_L2_P = np.zeros((len(range(0, 101)), len(range(0, 151))), dtype=float)

for i_N in range(0, 101, 1):
  print("Voy en: ", i_N)
  if i_N !=0:
    model = torch.load(DATA_ROOT + "modelo_completo_Ori.pth", weights_only=False)
    model_mod5 = copy.deepcopy(model)
    pruning_fraction_i = i_N/150
    # Aplicar pruning estructurado (por fila) a fc2
    prune.ln_structured(model_mod5.fc2, name='weight', amount=pruning_fraction_i, n=1, dim=0)
    with torch.no_grad():
        weight_mask = model_mod5.fc2.weight_mask
        if model_mod5.fc2.bias is not None:
            bias_mask = torch.ones_like(model_mod5.fc2.bias.data)
            for i in range(weight_mask.size(0)):
                if torch.all(weight_mask[i] == 0):  # Check if the entire row (neuron) is masked
                    bias_mask[i] = 0
            model_mod5.fc2.bias.data = bias_mask
    prune.remove(model_mod5.fc2, 'weight')
    model_mod5.eval()
    correct = 0
    total = 0
    with torch.no_grad():
      L_out, L4, L3, L2, L1, L_In = model_mod5(all_images) # Desempaquetar las salidas
      _, predicted = torch.max(L_out.data, 1)
      total += all_labels.size(0)
      correct += (predicted == all_labels).sum().item()
    Perf_L1_L2[i_N,j_N]= 100 * correct / total
  for j_N in range(0, 151, 1):
    if i_N == 0 and j_N == 0:
      model.eval()
      correct = 0
      total = 0
      with torch.no_grad():
        L_out, L4, L3, L2, L1, L_In = model(all_images) # Desempaquetar las salidas
        _, predicted = torch.max(L_out.data, 1)
        total += all_labels.size(0)
        correct += (predicted == all_labels).sum().item()
      Perf_L1_L2[i_N,j_N]= 100 * correct / total
    elif i_N == 0 and j_N != 0:
      model = torch.load(DATA_ROOT + "modelo_completo_Ori.pth", weights_only=False)
      model_mod5 = copy.deepcopy(model)
      pruning_fraction_j = j_N/300
      # Aplicar pruning estructurado (por fila) a fc2
      prune.ln_structured(model_mod5.fc1, name='weight', amount=pruning_fraction_j, n=1, dim=0)
      with torch.no_grad():
        weight_mask = model_mod5.fc1.weight_mask
        if model_mod5.fc1.bias is not None:
            bias_mask = torch.ones_like(model_mod5.fc1.bias.data)
            for i in range(weight_mask.size(0)):
                if torch.all(weight_mask[i] == 0):  # Check if the entire row (neuron) is masked
                    bias_mask[i] = 0
            model_mod5.fc1.bias.data = bias_mask
      prune.remove(model_mod5.fc1, 'weight')
      model_mod5.eval()
      correct = 0
      total = 0
      with torch.no_grad():
        L_out, L4, L3, L2, L1, L_In = model_mod5(all_images) # Desempaquetar las salidas
        _, predicted = torch.max(L_out.data, 1)
        total += all_labels.size(0)
        correct += (predicted == all_labels).sum().item()
      Perf_L1_L2[i_N,j_N]= 100 * correct / total
    else:
      model = torch.load(DATA_ROOT + "modelo_completo_Ori.pth", weights_only=False)
      model_mod5 = copy.deepcopy(model)
      prune.ln_structured(model_mod5.fc2, name='weight', amount=pruning_fraction_i, n=1, dim=0)
      prune.remove(model_mod5.fc2, 'weight')
      pruning_fraction_j = j_N/300
      # Aplicar pruning estructurado (por fila) a fc2
      prune.ln_structured(model_mod5.fc1, name='weight', amount=pruning_fraction_j, n=1, dim=0)
      with torch.no_grad():
        weight_mask = model_mod5.fc1.weight_mask
        if model_mod5.fc1.bias is not None:
            bias_mask = torch.ones_like(model_mod5.fc1.bias.data)
            for i in range(weight_mask.size(0)):
                if torch.all(weight_mask[i] == 0):  # Check if the entire row (neuron) is masked
                    bias_mask[i] = 0
            model_mod5.fc1.bias.data = bias_mask
      prune.remove(model_mod5.fc1, 'weight')
      model_mod5.eval()
      correct = 0
      total = 0
      with torch.no_grad():
        L_out, L4, L3, L2, L1, L_In = model_mod5(all_images) # Desempaquetar las salidas
        _, predicted = torch.max(L_out.data, 1)
        total += all_labels.size(0)
        correct += (predicted == all_labels).sum().item()
      Perf_L1_L2[i_N,j_N]= 100 * correct / total

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

x = np.linspace(0, Perf_L1_L2.shape[1] - 1, Perf_L1_L2.shape[1])
y = np.linspace(0, Perf_L1_L2.shape[0] - 1, Perf_L1_L2.shape[0])
X, Y = np.meshgrid(x, y)

surf = ax.plot_surface(X, Y, Perf_L1_L2, cmap='viridis')

# 4. Añadir etiquetas y título
ax.set_xlabel('N 300')
ax.set_ylabel('N 150')
ax.set_zlabel('Performance')

# 5. Añadir una barra de color (opcional)
#fig.colorbar(surf, shrink=0.5, aspect=10)

# Establecer los ángulos de visión
elev = 15   # Ángulo de elevación en grados (arriba/abajo)
azim = 20   # Ángulo de azimut en grados (izquierda/derecha)
ax.view_init(elev=elev, azim=azim)

# 6. Mostrar la gráfica
plt.show()

nombre_archivo = DATA_ROOT + 'perf_prun_pyTorch_L1.png'
plt.savefig(nombre_archivo, dpi=600, bbox_inches='tight')

model.fc1.weight.data.numpy()
plt.figure(figsize=(30, 50))
plt.imshow(model_mod5.fc2.weight.data.numpy(), cmap='gray')
plt.axis('off')  # Para no mostrar los ejes
plt.title('Pesos')
plt.show()

plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
plt.plot(model_mod5.fc2.bias.data.numpy(), marker='o', linestyle='-')
plt.grid(True)
plt.show()

In [ ]:
ruta_guardado_completo=DATA_ROOT + "modelo_completo_Ori.pth"

cont_i = 0
cont_j = 0
Perf_L1_L2_P = np.zeros((len(range(0, 101)), len(range(0, 151))), dtype=float)

L=1

# for i_N in range(0, 30, 1):
for i_N in range(0, 101, 1):
  print("Voy en: ", i_N)
  # for j_N in range(0, 50, 1):
  for j_N in range(0, 151, 1):
    if i_N == 0 and j_N == 0:
      model.eval()
      correct = 0
      total = 0
      with torch.no_grad():
        L_out, L4, L3, L2, L1, L_In = model(all_images) # Desempaquetar las salidas
        _, predicted = torch.max(L_out.data, 1)
        total += all_labels.size(0)
        correct += (predicted == all_labels).sum().item()
      Perf_L1_L2_P[cont_i,cont_j]= 100 * correct / total
    elif i_N == 0 and j_N != 0:
      model = torch.load(ruta_guardado_completo, weights_only=False)
      model_mod5 = copy.deepcopy(model)
      pruning_fraction_j = j_N/300
      # Aplicar pruning estructurado (por fila) a fc2
      prune.ln_structured(model_mod5.fc1, name='weight', amount=pruning_fraction_j, n=L, dim=0)
      with torch.no_grad():
        weight_mask = model_mod5.fc1.weight_mask
        if model_mod5.fc1.bias is not None:
            bias_mask = model_mod5.fc1.bias.data
            for i in range(weight_mask.size(0)):
                if torch.all(weight_mask[i] == 0):  # Check if the entire row (neuron) is masked
                    bias_mask[i] = 0
            model_mod5.fc1.bias.data = bias_mask
      prune.remove(model_mod5.fc1, 'weight')
      model_mod5.eval()
      correct = 0
      total = 0
      with torch.no_grad():
        L_out, L4, L3, L2, L1, L_In = model_mod5(all_images) # Desempaquetar las salidas
        _, predicted = torch.max(L_out.data, 1)
        total += all_labels.size(0)
        correct += (predicted == all_labels).sum().item()
      Perf_L1_L2_P[cont_i,cont_j]= 100 * correct / total
    else:
      model = torch.load(ruta_guardado_completo, weights_only=False)
      model_mod5 = copy.deepcopy(model)
      pruning_fraction_i = i_N/150
      prune.ln_structured(model_mod5.fc2, name='weight', amount=pruning_fraction_i, n=L, dim=0)
      with torch.no_grad():
        weight_mask = model_mod5.fc2.weight_mask
        if model_mod5.fc2.bias is not None:
          bias_mask = model_mod5.fc2.bias.data
          for i in range(weight_mask.size(0)):
            if torch.all(weight_mask[i] == 0):  # Check if the entire row (neuron) is masked
              bias_mask[i] = 0
          model_mod5.fc2.bias.data = bias_mask
      prune.remove(model_mod5.fc2, 'weight')
      pruning_fraction_j = j_N/300
      # Aplicar pruning estructurado (por fila) a fc2
      prune.ln_structured(model_mod5.fc1, name='weight', amount=pruning_fraction_j, n=L, dim=0)
      with torch.no_grad():
        weight_mask = model_mod5.fc1.weight_mask
        if model_mod5.fc1.bias is not None:
            bias_mask = model_mod5.fc1.bias.data
            for i in range(weight_mask.size(0)):
                if torch.all(weight_mask[i] == 0):  # Check if the entire row (neuron) is masked
                    bias_mask[i] = 0
            model_mod5.fc1.bias.data = bias_mask
      prune.remove(model_mod5.fc1, 'weight')
      model_mod5.eval()
      correct = 0
      total = 0
      with torch.no_grad():
        L_out, L4, L3, L2, L1, L_In = model_mod5(all_images) # Desempaquetar las salidas
        _, predicted = torch.max(L_out.data, 1)
        total += all_labels.size(0)
        correct += (predicted == all_labels).sum().item()
      Perf_L1_L2_P[cont_i,cont_j]= 100 * correct / total
    cont_j+=1
  cont_i+=1
  cont_j=0

from matplotlib import rcParams

rcParams.update({'font.size': 14})
from matplotlib.ticker import MultipleLocator

fig = plt.figure(figsize=(11, 12))
ax = fig.add_subplot(111, projection='3d')

x = np.linspace(0, Perf_L1_L2_P.shape[1] - 1, Perf_L1_L2_P.shape[1])
y = np.linspace(0, Perf_L1_L2_P.shape[0] - 1, Perf_L1_L2_P.shape[0])
X, Y = np.meshgrid(x, y)

surf = ax.plot_surface(X, Y, Perf_L1_L2_P, cmap='viridis')

# 4. Añadir etiquetas y título
ax.set_xlabel('N 300', labelpad=20)
ax.set_ylabel('N 150', labelpad=20)
ax.set_zlabel('Performance', labelpad=10)

ax.yaxis.set_major_locator(MultipleLocator(50))

# 5. Añadir una barra de color (opcional)
#fig.colorbar(surf, shrink=0.5, aspect=10)

# Establecer los ángulos de visión
elev = 15   # Ángulo de elevación en grados (arriba/abajo)
azim = 20   # Ángulo de azimut en grados (izquierda/derecha)
ax.view_init(elev=elev, azim=azim)

nombre_archivo = DATA_ROOT + 'perf_prun_CIFAR10_pyTorch_L'+str(L)+'.png'
plt.savefig(nombre_archivo, dpi=600, bbox_inches='tight')

# 6. Mostrar la gráfica
plt.show()

#model.fc1.weight.data.numpy()
#plt.figure(figsize=(30, 50))
#plt.imshow(model_mod5.fc2.weight.data.numpy(), cmap='gray')
#plt.axis('off')  # Para no mostrar los ejes
#plt.title('Pesos')
#plt.show()

#plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
#plt.plot(model_mod5.fc2.bias.data.numpy(), marker='o', linestyle='-')
#plt.grid(True)
#plt.show()

In [ ]:
import pickle


In [ ]:
import time
import torch.nn.utils.prune as prune

cont_i = 0
cont_j = 0
cont_k = 0
Perf_L1_L2_P_1 = np.zeros((len(range(0, 101)), len(range(0, 151)),10), dtype=float)
Time_L1_L2_P = np.zeros((len(range(0, 101)), len(range(0, 151)), 10), dtype=float)

for L in range(1,3):
  # for i_N in range(0, 30, 1):
  for i_N in range(0, 101, 1):
    print("Voy en: ", i_N)
    # for j_N in range(0, 50, 1):
    for j_N in range(0, 151, 1):
      for k in range(0,10):
        start_time = time.time()
        if i_N == 0 and j_N == 0:
          model.eval()
          correct = 0
          total = 0
          with torch.no_grad():
            L_out, L4, L3, L2, L1, L_In = model(all_images) # Desempaquetar las salidas
            _, predicted = torch.max(L_out.data, 1)
            total += all_labels.size(0)
            correct += (predicted == all_labels).sum().item()
          Perf_L1_L2_P_1[cont_i,cont_j,cont_k]= 100 * correct / total
        elif i_N == 0 and j_N != 0:
          model = torch.load(ruta_guardado_completo, weights_only=False)
          model_mod5 = copy.deepcopy(model)
          pruning_fraction_j = j_N/300
          # Aplicar pruning estructurado (por fila) a fc2
          prune.ln_structured(model_mod5.fc1, name='weight', amount=pruning_fraction_j, n=L, dim=0)
          with torch.no_grad():
            weight_mask = model_mod5.fc1.weight_mask
            if model_mod5.fc1.bias is not None:
                bias_mask = model_mod5.fc1.bias.data
                for i in range(weight_mask.size(0)):
                    if torch.all(weight_mask[i] == 0):  # Check if the entire row (neuron) is masked
                        bias_mask[i] = 0
                model_mod5.fc1.bias.data = bias_mask
          prune.remove(model_mod5.fc1, 'weight')
          model_mod5.eval()
          correct = 0
          total = 0
          with torch.no_grad():
            L_out, L4, L3, L2, L1, L_In = model_mod5(all_images) # Desempaquetar las salidas
            _, predicted = torch.max(L_out.data, 1)
            total += all_labels.size(0)
            correct += (predicted == all_labels).sum().item()
          Perf_L1_L2_P_1[cont_i,cont_j,cont_k]= 100 * correct / total
        else:
          model = torch.load(ruta_guardado_completo, weights_only=False)
          model_mod5 = copy.deepcopy(model)
          pruning_fraction_i = i_N/150
          prune.ln_structured(model_mod5.fc2, name='weight', amount=pruning_fraction_i, n=L, dim=0)
          with torch.no_grad():
            weight_mask = model_mod5.fc2.weight_mask
            if model_mod5.fc2.bias is not None:
              bias_mask = model_mod5.fc2.bias.data
              for i in range(weight_mask.size(0)):
                if torch.all(weight_mask[i] == 0):  # Check if the entire row (neuron) is masked
                  bias_mask[i] = 0
              model_mod5.fc2.bias.data = bias_mask
          prune.remove(model_mod5.fc2, 'weight')
          pruning_fraction_j = j_N/300
          # Aplicar pruning estructurado (por fila) a fc2
          prune.ln_structured(model_mod5.fc1, name='weight', amount=pruning_fraction_j, n=L, dim=0)
          with torch.no_grad():
            weight_mask = model_mod5.fc1.weight_mask
            if model_mod5.fc1.bias is not None:
                bias_mask = model_mod5.fc1.bias.data
                for i in range(weight_mask.size(0)):
                    if torch.all(weight_mask[i] == 0):  # Check if the entire row (neuron) is masked
                        bias_mask[i] = 0
                model_mod5.fc1.bias.data = bias_mask
          prune.remove(model_mod5.fc1, 'weight')
          model_mod5.eval()
          correct = 0
          total = 0
          with torch.no_grad():
            L_out, L4, L3, L2, L1, L_In = model_mod5(all_images) # Desempaquetar las salidas
            _, predicted = torch.max(L_out.data, 1)
            total += all_labels.size(0)
            correct += (predicted == all_labels).sum().item()
          Perf_L1_L2_P_1[cont_i,cont_j,cont_k]= 100 * correct / total
          Time_L1_L2_P[cont_i,cont_j,cont_k] = time.time() - start_time
        cont_k+=1
      # print(f'La media de duración fue: {np.mean(Time_L1_L2_P[cont_i,cont_j,:]):.4f} %')
      cont_j+=1
      cont_k=0
    cont_i+=1
    cont_j=0
  cont_i=0
  cont_j=0
  cont_k=0

  from matplotlib import rcParams
  from matplotlib.ticker import MultipleLocator

  rcParams.update({'font.size': 14})
  fig = plt.figure(figsize=(11, 12))
  ax = fig.add_subplot(111, projection='3d')

  x = np.linspace(0, Perf_L1_L2_P_1.shape[1] - 1, Perf_L1_L2_P_1.shape[1])
  y = np.linspace(0, Perf_L1_L2_P_1.shape[0] - 1, Perf_L1_L2_P_1.shape[0])
  X, Y = np.meshgrid(x, y)

  surf = ax.plot_surface(X, Y, Perf_L1_L2_P_1[:,:,3], cmap='viridis')

  # 4. Añadir etiquetas y título
  ax.set_xlabel('N 300', labelpad=20)
  ax.set_ylabel('N 150', labelpad=20)
  ax.set_zlabel('Performance', labelpad=10)

  ax.xaxis.set_major_locator(MultipleLocator(50))
  ax.yaxis.set_major_locator(MultipleLocator(25))

  # 5. Añadir una barra de color (opcional)
  #fig.colorbar(surf, shrink=0.5, aspect=10)

  # Establecer los ángulos de visión
  elev = 15   # Ángulo de elevación en grados (arriba/abajo)
  azim = 20   # Ángulo de azimut en grados (izquierda/derecha)
  ax.view_init(elev=elev, azim=azim)

  nombre_archivo = DATA_ROOT + 'perf_prun_CIFAR10_pyTorch_L'+str(L)+'.png'
  plt.savefig(nombre_archivo, dpi=300, bbox_inches='tight')

  # 6. Mostrar la gráfica
  plt.show()

  #root_N = DATA_ROOT + ''
  #nombre_archivo = "CIFAR_PyTorch_L"+str(L)+".pkl"

  #with open(root_N+nombre_archivo, 'wb') as archivo:
  #    pickle.dump([Perf_L1_L2_P_1, Time_L1_L2_P], archivo)

  #print(f"Variable guardada en {nombre_archivo}")

In [ ]:
model = torch.load(ruta_guardado_completo, weights_only=False)
model_mod5 = copy.deepcopy(model)

cont_i = 0
cont_j = 0
Perf_L1_L2 = np.zeros((len(range(0, -101, -1)), len(range(0, -151, -1))), dtype=float)

for i_N in range(0, -101, -1):
  print("Voy en: ", i_N)
  for j_N in range(0, -151, -1):

    if i_N == 0 and j_N == 0:
      model = torch.load(ruta_guardado_completo, weights_only=False)
      model.eval()
      correct = 0
      total = 0
      with torch.no_grad():
        L_out, L4, L3, L2, L1, L_In = model(all_images) # Desempaquetar las salidas
        _, predicted = torch.max(L_out.data, 1)
        total += all_labels.size(0)
        correct += (predicted == all_labels).sum().item()
      Perf_L1_L2[cont_i,cont_j]= 100 * correct / total
    elif i_N == 0:
      model = torch.load(ruta_guardado_completo, weights_only=False)
      model_mod5 = copy.deepcopy(model)
      model_mod5, Perf_L1_L2[cont_i,cont_j] = Quitar_Contri_NVerbo(model_mod5, indices_ordenamiento_L1[j_N:], 1)
    elif j_N == 0:
      model = torch.load(ruta_guardado_completo, weights_only=False)
      model_mod5 = copy.deepcopy(model)
      model_mod5, Perf_L1_L2[cont_i,cont_j] = Quitar_Contri_NVerbo(model_mod5, indices_ordenamiento[i_N:], 2)
    else:
      model = torch.load(ruta_guardado_completo, weights_only=False)
      model_mod5 = copy.deepcopy(model)
      model_mod5, _ = Quitar_Contri_NVerbo(model_mod5, indices_ordenamiento[i_N:], 2)
      model_mod5, Perf_L1_L2[cont_i,cont_j] = Quitar_Contri_NVerbo(model_mod5, indices_ordenamiento_L1[j_N:], 1)
    cont_j+=1
  cont_i+=1
  cont_j=0

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

x = np.linspace(0, Perf_L1_L2.shape[1] - 1, Perf_L1_L2.shape[1])
y = np.linspace(0, Perf_L1_L2.shape[0] - 1, Perf_L1_L2.shape[0])
X, Y = np.meshgrid(x, y)

surf = ax.plot_surface(X, Y, Perf_L1_L2, cmap='viridis')

# 4. Añadir etiquetas y título
ax.set_xlabel('N 300')
ax.set_ylabel('N 150')
ax.set_zlabel('Performance')

# 5. Añadir una barra de color (opcional)
#fig.colorbar(surf, shrink=0.5, aspect=10)

# Establecer los ángulos de visión
elev = 15   # Ángulo de elevación en grados (arriba/abajo)
azim = 20   # Ángulo de azimut en grados (izquierda/derecha)
ax.view_init(elev=elev, azim=azim)

# 6. Mostrar la gráfica
plt.show()

In [ ]:
model = torch.load(ruta_guardado_completo, weights_only=False)
model_mod5 = copy.deepcopy(model)

cont_i = 0
cont_j = 0
cont_k = 0
Perf_L1_L2_1 = np.zeros((len(range(0, -101, -1)), len(range(0, -151, -1)),10), dtype=float)
Time_L1_L2_NObSP = np.zeros((len(range(0, 101)), len(range(0, 151)), 10), dtype=float)

for i_N in range(0, -101, -1):
  print("Voy en: ", i_N)
  for j_N in range(0, -151, -1):
    for k_N in range(0,10):
      start_time = time.time()
      if i_N == 0 and j_N == 0:
        model = torch.load(ruta_guardado_completo, weights_only=False)
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
          L_out, L4, L3, L2, L1, L_In = model(all_images) # Desempaquetar las salidas
          _, predicted = torch.max(L_out.data, 1)
          total += all_labels.size(0)
          correct += (predicted == all_labels).sum().item()
        Perf_L1_L2_1[cont_i,cont_j,cont_k]= 100 * correct / total
      elif i_N == 0:
        model = torch.load(ruta_guardado_completo, weights_only=False)
        model_mod5 = copy.deepcopy(model)
        model_mod5, Perf_L1_L2_1[cont_i,cont_j,cont_k] = Quitar_Contri_NVerbo(model_mod5, indices_ordenamiento_L1[j_N:], 1)
      elif j_N == 0:
        model = torch.load(ruta_guardado_completo, weights_only=False)
        model_mod5 = copy.deepcopy(model)
        model_mod5, Perf_L1_L2_1[cont_i,cont_j,cont_k] = Quitar_Contri_NVerbo(model_mod5, indices_ordenamiento[i_N:], 2)
      else:
        model = torch.load(ruta_guardado_completo, weights_only=False)
        model_mod5 = copy.deepcopy(model)
        model_mod5, _ = Quitar_Contri_NVerbo(model_mod5, indices_ordenamiento[i_N:], 2)
        model_mod5, Perf_L1_L2_1[cont_i,cont_j,cont_k] = Quitar_Contri_NVerbo(model_mod5, indices_ordenamiento_L1[j_N:], 1)
      Time_L1_L2_NObSP[cont_i,cont_j,cont_k]= time.time() - start_time
      cont_k+=1
    cont_j+=1
    cont_k=0
  cont_i+=1
  cont_j=0

from matplotlib import rcParams
from matplotlib.ticker import MultipleLocator

rcParams.update({'font.size': 14})
fig = plt.figure(figsize=(11, 12))
ax = fig.add_subplot(111, projection='3d')

x = np.linspace(0, Perf_L1_L2_1.shape[1] - 1, Perf_L1_L2_1.shape[1])
y = np.linspace(0, Perf_L1_L2_1.shape[0] - 1, Perf_L1_L2_1.shape[0])
X, Y = np.meshgrid(x, y)

surf = ax.plot_surface(X, Y, Perf_L1_L2_1[:,:,8], cmap='viridis')

# 4. Añadir etiquetas y título
ax.set_xlabel('N 300', labelpad=20)
ax.set_ylabel('N 150', labelpad=20)
ax.set_zlabel('Performance', labelpad=10)

ax.xaxis.set_major_locator(MultipleLocator(50))
ax.yaxis.set_major_locator(MultipleLocator(25))

# 5. Añadir una barra de color (opcional)
#fig.colorbar(surf, shrink=0.5, aspect=10)

# Establecer los ángulos de visión
elev = 15   # Ángulo de elevación en grados (arriba/abajo)
azim = 20   # Ángulo de azimut en grados (izquierda/derecha)
ax.view_init(elev=elev, azim=azim)

nombre_archivo = DATA_ROOT + 'perf_prun_CIFARF10_NObSP.png'
plt.savefig(nombre_archivo, dpi=300, bbox_inches='tight')

# 6. Mostrar la gráfica
plt.show()

#root_N = DATA_ROOT + ''
#nombre_archivo = "CIFARF10_NObSP.pkl"

#with open(root_N+nombre_archivo, 'wb') as archivo:
#    pickle.dump([Perf_L1_L2_1, Time_L1_L2_NObSP], archivo)

#print(f"Variable guardada en {nombre_archivo}")

In [ ]:
Time_L1_L2_NObSP=Time_L1_L2_P

In [ ]:
pruning_fraction = 0.2  # Fracción de neuronas a podar
1/300

# Aplicar pruning estructurado (por fila) a fc1
prune.ln_structured(model.fc1, name='weight', amount=pruning_fraction, n=2, dim=0)

# Aplicar pruning estructurado (por fila) a fc2
prune.ln_structured(model.fc2, name='weight', amount=pruning_fraction, n=2, dim=0)

# Hacer que los buffers de máscara sean persistentes (opcional, para guardado del modelo)
prune.remove(model.fc1, 'weight')


pruning_fraction = 0.1
for name, module in model.named_modules():
  if isinstance(module, nn.Linear):
    if module.out_features > 1:
      print(name)
      print(module)
      print(module.out_features)
      neuron_norms = torch.norm(module.weight, p=2, dim=1)
      print(neuron_norms)
      num_prune = int(pruning_fraction * module.out_features)
      print(num_prune)
      if num_prune > 0:
        _, indices_to_prune = torch.topk(neuron_norms.abs(), k=num_prune, largest=False)
      print(indices_to_prune)
def apply_neuron_pruning_magnitude(model: nn.Module, pruning_fraction: float):
    """Applies magnitude-based neuron pruning to linear layers."""
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            if module.out_features > 1:  # Avoid pruning the output layer too much
                neuron_norms = torch.norm(module.weight, p=2, dim=1)
                num_prune = int(pruning_fraction * module.out_features)
                if num_prune > 0:
                    _, indices_to_prune = torch.topk(neuron_norms.abs(), k=num_prune, largest=False)
                    for index in indices_to_prune:
                        prune_linear_neuron(module, index)

In [ ]:
# Evaluar el modelo
model.eval()
correct = 0
total = 0
with torch.no_grad():
  L_out, L4, L3, L2, L1, L_In = model(all_images) # Desempaquetar las salidas
  _, predicted = torch.max(L_out.data, 1)
  total += all_labels.size(0)
  correct += (predicted == all_labels).sum().item()

print(f'\nPrecisión del modelo en el conjunto de prueba: {100 * correct / total:.2f} %')

In [ ]:
# Evaluar el modelo
model_mod.eval()
correct = 0
total = 0
with torch.no_grad():
  L_out, L4, L3, L2, L1, L_In = model_mod(all_images) # Desempaquetar las salidas
  _, predicted = torch.max(L_out.data, 1)
  total += all_labels.size(0)
  correct += (predicted == all_labels).sum().item()

print(f'\nPrecisión del modelo en el conjunto de prueba: {100 * correct / total:.2f} %')

In [ ]:
print("Se quita el indice: ",indices_ordenamiento[-2])

first_linear_layer = model_mod.fc2
weights_layer1 = first_linear_layer.weight.data.numpy()
biases_layer1 = first_linear_layer.bias.data.numpy()

print("\nForma de los pesos de la primera capa lineal (150 neuronas):", weights_layer1.shape)
print(weights_layer1[indices_ordenamiento[-2]-4:indices_ordenamiento[-2]+1,:])
print("Forma de los biases de la primera capa lineal (150 neuronas):", biases_layer1.shape)
plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
plt.plot(biases_layer1, marker='o', linestyle='-')
plt.grid(True)
plt.show()

first_linear_layer = model_mod.fc2
weights_layer1 = first_linear_layer.weight.data.numpy()
weights_layer1[indices_ordenamiento[-2],:] = 0
biases_layer1 = first_linear_layer.bias.data.numpy()
biases_layer1[indices_ordenamiento[-2]] = 0

model_mod.fc2.weight.data = torch.from_numpy(weights_layer1)
model_mod.fc2.bias.data = torch.from_numpy(biases_layer1)

print("\nForma de los pesos de la primera capa lineal (150 neuronas):", weights_layer1.shape)
print(weights_layer1[indices_ordenamiento[-2]-4:indices_ordenamiento[-2]+1,:])
print("Forma de los biases de la primera capa lineal (150 neuronas):", biases_layer1.shape)
plt.figure(figsize=(12, 6))  # Ajustar el tamaño de la figura si es necesario
plt.plot(biases_layer1, marker='o', linestyle='-')
plt.grid(True)
plt.show()

## 10. Cost and figuresTiming of both criteria, and the figures reported in the paper.

In [ ]:
import time

# Sección de código que quieres medir



for Perf_N in range(20,50,10):
  Quitar=Perf_N
  for L_n in range(1,3):

    start_time = time.time()

    Perf=[]

    model = torch.load(ruta_guardado_completo, weights_only=False)

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
      L_out, L4, L3, L2, L1, L_In = model(all_images) # Desempaquetar las salidas
      _, predicted = torch.max(L_out.data, 1)
      total += all_labels.size(0)
      correct += (predicted == all_labels).sum().item()
    print("Desempeño del modelo original: ",100 * correct / total)
    Perf.append(100 * correct / total)

    model_mod4 = copy.deepcopy(model)
    model_mod4, _  = Quitar_Contri_NVerbo(model_mod4, indices_ordenamiento_L1[-Quitar:], 1)
    model_mod4, Perf_NObSP = Quitar_Contri_NVerbo(model_mod4, indices_ordenamiento[-Quitar:], 2)

    model = torch.load(ruta_guardado_completo, weights_only=False)
    model_mod5 = copy.deepcopy(model)
    L=L_n
    pruning_fraction_i = Quitar/150
    neuron_norms_L2 = torch.norm(model_mod5.fc2.weight, p=L, dim=1)
    prune.ln_structured(model_mod5.fc2, name='weight', amount=pruning_fraction_i, n=L, dim=0)
    with torch.no_grad():
      weight_mask = model_mod5.fc2.weight_mask
      if model_mod5.fc2.bias is not None:
        bias_mask_L2 = model_mod5.fc2.bias.data
        for i in range(weight_mask.size(0)):
          if torch.all(weight_mask[i] == 0):  # Check if the entire row (neuron) is masked
            bias_mask_L2[i] = 0
        model_mod5.fc2.bias.data = bias_mask_L2

    prune.remove(model_mod5.fc2, 'weight')
    pruning_fraction_j = Quitar/300
    # Aplicar pruning estructurado (por fila) a fc2
    neuron_norms_L1 = torch.norm(model_mod5.fc1.weight, p=L, dim=1)
    prune.ln_structured(model_mod5.fc1, name='weight', amount=pruning_fraction_j, n=L, dim=0)
    with torch.no_grad():
      weight_mask = model_mod5.fc1.weight_mask
      if model_mod5.fc1.bias is not None:
        bias_mask_L1 = model_mod5.fc1.bias.data
        for i in range(weight_mask.size(0)):
          if torch.all(weight_mask[i] == 0):  # Check if the entire row (neuron) is masked
            bias_mask_L1[i] = 0
        model_mod5.fc1.bias.data = bias_mask_L1
    prune.remove(model_mod5.fc1, 'weight')
    model_mod5.eval()
    correct = 0
    total = 0
    with torch.no_grad():
      L_out, L4, L3, L2, L1, L_In = model_mod5(all_images) # Desempaquetar las salidas
      _, predicted = torch.max(L_out.data, 1)
      total += all_labels.size(0)
      correct += (predicted == all_labels).sum().item()

    # NObSP
    print("###### CON NObSP ######")
    print("Desempeño del modelo: ", Perf_NObSP)
    Perf.append(Perf_NObSP)
    print("Neuronas de la L1 que quita NObSP: ", np.sort(indices_ordenamiento_L1[-Quitar:]))
    print("Valores de contribución de L1 que quita NObSP: ", concurrencia_matrix_L1[np.sort(indices_ordenamiento_L1[-Quitar:])])
    print("Valores de neuronas de L1 que quita PyTorch: ", neuron_norms_L1[np.sort(indices_ordenamiento_L1[-Quitar:])])
    print("Neuronas de la L2 que quita NObSP: ", np.sort(indices_ordenamiento[-Quitar:]))
    print("Valores de contribución de L2 que quita NObSP: ", concurrencia_matrix_L2[np.sort(indices_ordenamiento[-Quitar:])])
    print("Valores de neuronas de L2 que quita PyTorch: ", neuron_norms_L2[np.sort(indices_ordenamiento[-Quitar:])])

    # PyTorch
    print("###### CON PyTorch ######")
    print("Desempeño del modelo: ",100 * correct / total)
    Perf.append(100 * correct / total)
    print("Neuronas de la L1 que quita PyTorch: ", np.where(bias_mask_L1==0)[0])
    print("Valores de las neuronas de L1 que quita PyTorch: ",neuron_norms_L1[np.where(bias_mask_L1==0)[0]])
    print("Valores de contribución de L1 que quita NObSP: ", concurrencia_matrix_L1[np.where(bias_mask_L1==0)[0]])
    print("Neuronas de la L2 que quita NObSP: ", np.where(bias_mask_L2==0)[0])
    print("Valores de las neuronas de L2 que quita PyTorch: ",neuron_norms_L2[np.where(bias_mask_L2==0)[0]])
    print("Valores de contribución de L2 que quita NObSP: ", concurrencia_matrix_L2[np.where(bias_mask_L2==0)[0]])

    array=np.column_stack((np.sort(indices_ordenamiento_L1[-Quitar:]),
                    concurrencia_matrix_L1[np.sort(indices_ordenamiento_L1[-Quitar:])],
                    neuron_norms_L1[np.sort(indices_ordenamiento_L1[-Quitar:])].detach().numpy(),
                    np.sort(indices_ordenamiento[-Quitar:]),
                    concurrencia_matrix_L2[np.sort(indices_ordenamiento[-Quitar:])],
                    neuron_norms_L2[np.sort(indices_ordenamiento[-Quitar:])].detach().numpy()))

    columnas = ['Indice_L1', 'Contribucion', 'Norma', 'Indice_L2', 'Contribucion', 'Norma']
    df = pd.DataFrame(array, columns=columnas)

    array1=np.column_stack((np.where(bias_mask_L1==0)[0],
                            neuron_norms_L1[np.where(bias_mask_L1==0)[0]].detach().numpy(),
                            concurrencia_matrix_L1[np.where(bias_mask_L1==0)[0]],
                            np.where(bias_mask_L2==0)[0],
                            neuron_norms_L2[np.where(bias_mask_L2==0)[0]].detach().numpy(),
                            concurrencia_matrix_L2[np.where(bias_mask_L2==0)[0]]))

    # Convertir a DataFrame
    columnas = ['Indice_L1', 'Contribucion', 'Norma', 'Indice_L2', 'Contribucion', 'Norma']
    df1 = pd.DataFrame(array1, columns=columnas)

    end_time = time.time()
    elapsed_time = end_time - start_time

    print(f"Tiempo de ejecución: {elapsed_time:.4f} segundos")

    Neuro_concat_L1 = np.unique(np.concatenate((np.sort(indices_ordenamiento_L1[-Quitar:]), np.where(bias_mask_L1==0)[0])))
    Neuro_concat_L2 = np.unique(np.concatenate((np.sort(indices_ordenamiento[-Quitar:]), np.where(bias_mask_L2==0)[0])))

    model = torch.load(ruta_guardado_completo, weights_only=False)
    model_mod4 = copy.deepcopy(model)
    model_mod4, _  = Quitar_Contri_NVerbo(model_mod4, Neuro_concat_L1, 1)
    model_mod4, Perf_NObSP = Quitar_Contri_NVerbo(model_mod4, Neuro_concat_L2, 2)
    print("###### CON NObSP-PyTorch ######")
    print("Desempeño del modelo: ", Perf_NObSP)
    Perf.append(Perf_NObSP)
    print(f"Elimino de la capa 1: {len(Neuro_concat_L1)} y de la capa 2: {len(Neuro_concat_L2)}")
    Perf.append(len(Neuro_concat_L1))
    Perf.append(len(Neuro_concat_L2))
    model = torch.load(ruta_guardado_completo, weights_only=False)
    model_mod4 = copy.deepcopy(model)
    model_mod4, _  = Quitar_Contri_NVerbo(model_mod4, indices_ordenamiento_L1[-len(Neuro_concat_L1):], 1)
    model_mod4, Perf_NObSP = Quitar_Contri_NVerbo(model_mod4, indices_ordenamiento[-len(Neuro_concat_L2):], 2)
    print("###### CON NObSP ######")
    print("Desempeño del modelo: ", Perf_NObSP)
    Perf.append(Perf_NObSP)

    model = torch.load(ruta_guardado_completo, weights_only=False)
    model_mod5 = copy.deepcopy(model)
    pruning_fraction_i = len(Neuro_concat_L2)/150
    neuron_norms_L2 = torch.norm(model_mod5.fc2.weight, p=L, dim=1)
    prune.ln_structured(model_mod5.fc2, name='weight', amount=pruning_fraction_i, n=L, dim=0)
    with torch.no_grad():
      weight_mask = model_mod5.fc2.weight_mask
      if model_mod5.fc2.bias is not None:
        bias_mask_L2 = model_mod5.fc2.bias.data
        for i in range(weight_mask.size(0)):
          if torch.all(weight_mask[i] == 0):  # Check if the entire row (neuron) is masked
            bias_mask_L2[i] = 0
        model_mod5.fc2.bias.data = bias_mask_L2
    prune.remove(model_mod5.fc2, 'weight')

    pruning_fraction_j = len(Neuro_concat_L1)/300
    # Aplicar pruning estructurado (por fila) a fc2
    neuron_norms_L1 = torch.norm(model_mod5.fc1.weight, p=L, dim=1)
    prune.ln_structured(model_mod5.fc1, name='weight', amount=pruning_fraction_j, n=L, dim=0)
    with torch.no_grad():
      weight_mask = model_mod5.fc1.weight_mask
      if model_mod5.fc1.bias is not None:
        bias_mask_L1 = model_mod5.fc1.bias.data
        for i in range(weight_mask.size(0)):
          if torch.all(weight_mask[i] == 0):  # Check if the entire row (neuron) is masked
            bias_mask_L1[i] = 0
        model_mod5.fc1.bias.data = bias_mask_L1
    prune.remove(model_mod5.fc1, 'weight')
    model_mod5.eval()
    correct = 0
    total = 0
    with torch.no_grad():
      L_out, L4, L3, L2, L1, L_In = model_mod5(all_images) # Desempaquetar las salidas
      _, predicted = torch.max(L_out.data, 1)
      total += all_labels.size(0)
      correct += (predicted == all_labels).sum().item()

    print("###### CON PyTorch ######")
    print("Desempeño del modelo: ",100 * correct / total)
    Perf.append(100 * correct / total)

    columnas = ['Perf_Ori','Perf_NObSP','Perf_PyTorch','Perf_Combi','w/o N1','w/o N2','Perf_NObSP_C','Perf_Pytorch_C']
    df2 = pd.DataFrame(np.array(Perf).reshape(1, 8), columns=columnas)

    with pd.ExcelWriter('resultado_L'+str(L)+'_N_'+str(Quitar)+'.xlsx', engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name='wNObSP', index=False)
        df1.to_excel(writer, sheet_name='wPyTorch', index=False)
        df2.to_excel(writer, sheet_name='Performance', index=False)

    from google.colab import files
    files.download('resultado_L'+str(L)+'_N_'+str(Quitar)+'.xlsx')

In [ ]:
import pickle

root_N = DATA_ROOT + ''
nombre_archivo = "CIFAR_PyTorch_L1.pkl"

# Especifica el nombre del archivo pickle que quieres cargar
nombre_archivo = root_N+nombre_archivo  # Reemplaza con el nombre de tu archivo

try:
    with open(nombre_archivo, 'rb') as archivo:
        Perf_L1_L2_P1, Time_L1 = pickle.load(archivo)
    print(f"Variable cargada exitosamente desde {nombre_archivo}")
    # Ahora puedes trabajar con la variable_cargada
    # Por ejemplo, si guardaste una lista:
    # print(variable_cargada[0])
    # Si guardaste un diccionario:
    # print(variable_cargada['clave'])
except FileNotFoundError:
    print(f"Error: El archivo {nombre_archivo} no fue encontrado.")
except Exception as e:
    print(f"Ocurrió un error al cargar el archivo pickle: {e}")

nombre_archivo = "CIFAR_PyTorch_L2.pkl"

# Especifica el nombre del archivo pickle que quieres cargar
nombre_archivo = root_N+nombre_archivo  # Reemplaza con el nombre de tu archivo

try:
    with open(nombre_archivo, 'rb') as archivo:
        Perf_L1_L2_P1, Time_L2 = pickle.load(archivo)
    print(f"Variable cargada exitosamente desde {nombre_archivo}")
    # Ahora puedes trabajar con la variable_cargada
    # Por ejemplo, si guardaste una lista:
    # print(variable_cargada[0])
    # Si guardaste un diccionario:
    # print(variable_cargada['clave'])
except FileNotFoundError:
    print(f"Error: El archivo {nombre_archivo} no fue encontrado.")
except Exception as e:
    print(f"Ocurrió un error al cargar el archivo pickle: {e}")

nombre_archivo = "CIFARF10_NObSP.pkl"

# Especifica el nombre del archivo pickle que quieres cargar
nombre_archivo = root_N+nombre_archivo  # Reemplaza con el nombre de tu archivo

try:
    with open(nombre_archivo, 'rb') as archivo:
        Perf_L1_L2_P1, Time_NOb = pickle.load(archivo)
    print(f"Variable cargada exitosamente desde {nombre_archivo}")
    # Ahora puedes trabajar con la variable_cargada
    # Por ejemplo, si guardaste una lista:
    # print(variable_cargada[0])
    # Si guardaste un diccionario:
    # print(variable_cargada['clave'])
except FileNotFoundError:
    print(f"Error: El archivo {nombre_archivo} no fue encontrado.")
except Exception as e:
    print(f"Ocurrió un error al cargar el archivo pickle: {e}")

In [ ]:
fig = plt.figure(figsize=(50, 100), constrained_layout=True)
ax = fig.add_subplot(131, projection='3d')
ax1 = fig.add_subplot(132, projection='3d')
ax2 = fig.add_subplot(133, projection='3d')

x = np.linspace(0, Time_L1.shape[1] - 1, Time_L1.shape[1])
y = np.linspace(0, Time_L1.shape[0] - 1, Time_L1.shape[0])
X, Y = np.meshgrid(x, y)

ax.plot_surface(X, Y, np.mean(Time_L1,2), cmap='viridis')
ax1.plot_surface(X, Y, np.mean(Time_L2,2), cmap='viridis')
ax2.plot_surface(X, Y, np.mean(Time_NOb,2), cmap='viridis')

Fsize=24

# 4. Añadir etiquetas y título
ax.set_xlabel('N 300',fontsize=Fsize, labelpad=20)
ax.set_ylabel('N 150',fontsize=Fsize, labelpad=20)
ax.set_zlabel('L1',fontsize=Fsize, labelpad=30)
ax.view_init(elev=15)

# 4. Añadir etiquetas y título
ax1.set_xlabel('N 300',fontsize=Fsize, labelpad=20)
ax1.set_ylabel('N 150',fontsize=Fsize, labelpad=20)
ax1.set_zlabel('L2',fontsize=Fsize, labelpad=20)
ax1.view_init(elev=15)

# 4. Añadir etiquetas y título
ax2.set_xlabel('N 300',fontsize=Fsize, labelpad=20)
ax2.set_ylabel('N 150',fontsize=Fsize, labelpad=20)
ax2.set_zlabel('NObSP',fontsize=Fsize, labelpad=20)
ax2.view_init(elev=15)

plt.xticks(fontsize=Fsize)
plt.yticks(fontsize=Fsize)

# 5. Añadir una barra de color (opcional)
#fig.colorbar(surf, shrink=0.5, aspect=10)

# # Establecer los ángulos de visión
# elev = 15   # Ángulo de elevación en grados (arriba/abajo)
# azim = 20   # Ángulo de azimut en grados (izquierda/derecha)
# ax.view_init(elev=elev, azim=azim)

# 6. Mostrar la gráfica
plt.show()

In [ ]:
v_T_L1 = Time_L1.reshape(-1)
v_T_L2 = Time_L1.reshape(-1)
v_T_NOb = Time_NOb.reshape(-1)

for i in range(25,100,25):
  print("L1: ",np.percentile(v_T_L1, i))
  print("L2: ",np.percentile(v_T_L2, i))
  print("NObSP: ",np.percentile(v_T_NOb, i))

In [ ]:
# Evaluar el modelo
model.eval()
correct = 0
total = 0
with torch.no_grad():
  L_out, L4, L3, L2, L1, L_In = model(all_images) # Desempaquetar las salidas
  _, predicted = torch.max(L_out.data, 1)
  total += all_labels.size(0)
  correct += (predicted == all_labels).sum().item()

print(f'\nPrecisión del modelo en el conjunto de prueba: {100 * correct / total:.2f} %')

In [ ]:
# Evaluar el modelo
model_mod.eval()
correct = 0
total = 0
with torch.no_grad():
  L_out, L4, L3, L2, L1, L_In = model_mod(all_images) # Desempaquetar las salidas
  _, predicted = torch.max(L_out.data, 1)
  total += all_labels.size(0)
  correct += (predicted == all_labels).sum().item()

print(f'\nPrecisión del modelo en el conjunto de prueba: {100 * correct / total:.2f} %')

In [ ]:
# Eliminar la segunda dimensión (el valor de contribución)
Cont_NL1_reduced = Cont_NL1_sorted[:, 0, :]  # Seleccionar todos los elementos en la dimensión 0 y 2, y el elemento 1 en la dimensión 1

# Crear la matriz de concurrencia
concurrencia_matrix = np.zeros((300, 300), dtype=int)

# Llenar la matriz de concurrencia
for neu_i in range(300):
    for pos_i in range(300):
        # Contar cuántas veces la neurona neu_i aparece en la posición pos_i para todas las clases
        concurrencia_matrix[pos_i,neu_i] = torch.sum(Cont_NL1_reduced[pos_i] == neu_i+1).item()

print("Forma de la matriz de concurrencia:", concurrencia_matrix.shape)
print("Ejemplo de la matriz de concurrencia:\n", concurrencia_matrix[:,0])  # Imprime un fragmento de la matriz

plt.imshow(concurrencia_matrix, cmap='gray')
plt.axis('off')  # Para no mostrar los ejes
plt.title('Posición de las neuronas')
plt.show()

In [ ]:
neu_i=1
Cont_NL1_sorted[Cont_NL1_sorted[:,0,0] == 3,1,0].item()
Cont_NL1_reduced

In [ ]:
!cat /proc/cpuinfo
!cat /proc/meminfo